<a href="https://colab.research.google.com/github/sruthi-analyst/sruthi-codeboosters-2026/blob/main/Day9/Day_9_task.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install groq -q

print("Libraries installed successfully!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 4.8 MB/s eta 0:00:00
Libraries installed successfully!


In [3]:
import sqlite3
import os
import pandas as pd
from groq import Groq
import re

print("All libraries imported successfully")

All libraries imported successfully


In [4]:
import os
from google.colab import userdata

try:
    os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
    print("API Key successfully loaded from Colab Secrets!")
except userdata.SecretNotFoundError:
    print("Error: Please add 'GROQ_API_KEY' to your Colab Secrets sidebar.")

API Key successfully loaded from Colab Secrets!


In [5]:
client = Groq(api_key=os.environ["GROQ_API_KEY"])
print("Groq client initialized successfully!")

MODEL = "llama-3.1-8b-instant"

print("groq client initialised")
print(f"Using model: {MODEL}")

Groq client initialized successfully!
groq client initialised
Using model: llama-3.1-8b-instant


In [6]:
csv_data = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/datasets/student_performance.csv")
print("Imported Dataset into DataFrame successfully!")

Imported Dataset into DataFrame successfully!


In [7]:
conn = sqlite3.connect("study_statistics.db")
csv_data.to_sql("students", conn, if_exists="replace", index=False)
print("SQLite database loaded with Dataset successfully!")

SQLite database loaded with Dataset successfully!


In [8]:
def get_schema(conn, table_name="students"):
  """  Reads structure of database...  """
  cursor = conn.cursor()
  cursor.execute(f"PRAGMA table_info({table_name})")
  columns = cursor.fetchall()
  schema_lines = [f"Table: {table_name}"]
  schema_lines.append("Columns: ")

  for col in columns:
    schema_lines.append(f"  - {col[1]} ({col[2]})")

  cursor.execute(f"SELECT * FROM {table_name} LIMIT 3")
  sample_data = cursor.fetchall()
  schema_lines.append("\nSample rows (first 3): ")

  for row in sample_data:
    schema_lines.append(f"  {row}")

  return "\n".join(schema_lines)

In [9]:
def generate_sql(user_question, schema_text, client, model):
  """ Generates a SQL query that answers the user's question. """

  system_prompt = f"""You are the developer of SQL.
  You are connected to a SQLite database with the following structure:
  {schema_text}

  Rules you must follow:
  1. Generate ONLY a valid SQLite SQL query.
  2. Do not include any explanation or text — only the SQL query.
  3. Do not use markdown code blocks. Return the raw SQL only.
  4. The table name is: students
  5. Only use column names that exist in the schema above.
  6. Use single quotes for string values in WHERE clauses (example: WHERE subject = 'Programming').
  7. If the user asks for top N, use ORDER BY marks DESC LIMIT N.
  """

  response = client.chat.completions.create(
      model=model,
      messages=[
          {"role": "system", "content": system_prompt},
          {"role": "user", "content": user_question}
      ],
      temperature = 0.0
  )

  sql_query = response.choices[0].message.content.strip()
  return sql_query


In [10]:
def execute_sql(sql_query, conn):
  """ this is an sql query executing function """
  clean_sql = sql_query.strip()
  clean_sql = re.sub(r'```\s*','', clean_sql)
  clean_sql = sql_query.strip()

  try:
    result_df = pd.read_sql_query(clean_sql, conn)
    return result_df, None
  except Exception as e:
    return None, str(e)

In [11]:
def to_sql_agent(user_question, conn, client, model, verbose=True):
  """ this is main AgentAI function """
  print(f"User question:{user_question}")
  print("="*60)
  if verbose:
    print("\n[STEP 1] Reading database schema")
  schema_text=get_schema(conn)

  if verbose:
    print("Schema loaded succssfully")

  if verbose:
    print("\n[STEP 2] Generating SQL query with groq LLM...")

  generated_sql=generate_sql(user_question,schema_text,client,model)
  if verbose:
    print(f'Generated SQL:\n{generated_sql}')

  if verbose:
    print("\n[STEP 3] Executing SQL on db...")

  result_df,error = execute_sql(generated_sql,conn)

  if error:
    print(f'Error executing SQL:{error}')
    return None,generated_sql

  if verbose:
    print(f'\n[STEP 4] QUERY RETURNED {len(result_df)} row(s)')
    print("\nRESULTS:")
    print("-"*60)
    print(result_df.to_string(index=False))

  print("="*60)
  return result_df,generated_sql

In [41]:
#TEST EXECUTION 1
question = "List all students scored >90 in maths and programming"
result, sql_used = to_sql_agent(question, conn, client, MODEL, verbose=True)
# schema = get_schema(conn)
# sql_topper = generate_sql(question, schema, client, MODEL)
# print(f"\nGenerated SQL:\n{sql_topper}")

# print(f"Execute SQL: {sql_topper}")
# result_topper, error_topper = execute_sql(sql_topper, conn)

User question:List all students scored >90 in maths and programming

[STEP 1] Reading database schema
Schema loaded succssfully

[STEP 2] Generating SQL query with groq LLM...
Generated SQL:
SELECT * FROM students WHERE math_score > 90 AND programming_score > 90

[STEP 3] Executing SQL on db...

[STEP 4] QUERY RETURNED 4 row(s)

RESULTS:
------------------------------------------------------------
 student_id           name  age gender       department  semester  math_score  science_score  english_score  programming_score  attendance_percentage    city  admission_year
       1005     Arjun Nair   19   Male Computer Science         2          92             88             81                 95                     90   Kochi            2023
       1010     Ananya Das   19 Female Computer Science         2          95             89             90                 97                     98 Kolkata            2023
       1022    Tanvi Mehta   19 Female Computer Science         2          93

In [40]:
def to_nlp_agent(user_question, conn, client, model, verbose=True):
  """ Generates a test response that answers the user's question in Natural Language. """
  result_nlp, gen_sql = to_sql_agent(user_question, conn, client, model, verbose)
  system_prompt = f"""You are a straightforward precise perfect language assitant.
  You are given a DataFrame of {result} answering {user_question}

  Rules you must follow:
  1. Start the response with {user_question} in first person narration, past tense, passive voice.
  Do not mension about the rephrased version
  2. Do not include any explanation unless asked for explicitly.
  3. Return as a list or paragraph only.
  4. Do not return as DataFrame.
  5. Only use column names that exist in the result above.
  6. Start the response by mensioning rephrased version of {user_question}.
  """

  response = client.chat.completions.create(
      model=model,
      messages=[
          {"role": "system", "content": system_prompt},
          {"role": "user", "content": user_question}
      ],
      temperature = 0.3
  )

  nlp_response = response.choices[0].message.content.strip()
  if verbose:
    print(f'\n[STEP 5] NLP ANSWER')
    print("\nRESPONSE:")
    print("-"*60)
    print(nlp_response)
  return nlp_response

In [42]:
#TEST EXECUTION 2
question = "List all students scored >90 in maths and programming"
nlp_response = to_nlp_agent(question, conn, client, MODEL, verbose=True)

User question:List all students scored >90 in maths and programming

[STEP 1] Reading database schema
Schema loaded succssfully

[STEP 2] Generating SQL query with groq LLM...
Generated SQL:
SELECT * FROM students WHERE math_score > 90 AND programming_score > 90

[STEP 3] Executing SQL on db...

[STEP 4] QUERY RETURNED 4 row(s)

RESULTS:
------------------------------------------------------------
 student_id           name  age gender       department  semester  math_score  science_score  english_score  programming_score  attendance_percentage    city  admission_year
       1005     Arjun Nair   19   Male Computer Science         2          92             88             81                 95                     90   Kochi            2023
       1010     Ananya Das   19 Female Computer Science         2          95             89             90                 97                     98 Kolkata            2023
       1022    Tanvi Mehta   19 Female Computer Science         2          93